In [ ]:
from utils import load_data, apply_smote_per_group, get_features_root


In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef
from sklearn.model_selection import StratifiedGroupKFold
from matplotlib.patches import Patch


In [ ]:
device = "emotibit_volar"
window = "8"
overlap_mode = "con_solapamiento"
smote_method = "random"

n_splits = 10
threshold_start = 0.0000
threshold_end = 0.0101
threshold_step = 0.0001
threshold_n_jobs = max(1, (os.cpu_count() or 1) - 1)


In [ ]:
X, y, groups = load_data(device=device, window=window, overlap_mode=overlap_mode)
print(f"length of X: {len(X)}, length of y: {len(y)}, length of groups: {len(groups)}")
print(y.value_counts())


In [ ]:
def evaluate_single_threshold(
    threshold,
    X,
    y,
    groups,
    smote_method=False,
    n_splits=10,
):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

    accuracy_scores = []
    f1_scores = []
    f1_macro_scores = []
    balanced_accuracy_scores = []
    mcc_scores = []
    n_features_per_fold = []
    skipped_folds = 0

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]

        if smote_method:
            X_train_sel, y_train_sel = apply_smote_per_group(
                X_train,
                y_train,
                groups_train,
                method=smote_method,
            )
        else:
            X_train_sel, y_train_sel = X_train, y_train

        selector = RandomForestClassifier(
            n_estimators=500,
            n_jobs=1,
            random_state=42,
        )
        selector.fit(X_train_sel, y_train_sel)

        fold_importance = pd.Series(
            selector.feature_importances_,
            index=X_train_sel.columns,
        )

        feature_list = fold_importance[fold_importance > threshold].index.tolist()
        n_features_per_fold.append(len(feature_list))

        if len(feature_list) == 0:
            skipped_folds += 1
            continue

        model = RandomForestClassifier(
            n_estimators=500,
            n_jobs=1,
            random_state=42,
        )
        model.fit(X_train_sel[feature_list], y_train_sel)
        y_pred = model.predict(X_test[feature_list])

        accuracy_scores.append(accuracy_score(y_test, y_pred))
        f1_scores.append(
            f1_score(y_test, y_pred, average="weighted", zero_division=0)
        )
        f1_macro_scores.append(
            f1_score(y_test, y_pred, average="macro", zero_division=0)
        )
        balanced_accuracy_scores.append(balanced_accuracy_score(y_test, y_pred))
        mcc_scores.append(matthews_corrcoef(y_test, y_pred))

    return {
        "threshold": float(threshold),
        "n_features_mean": float(np.mean(n_features_per_fold)) if n_features_per_fold else 0.0,
        "n_features_std": float(np.std(n_features_per_fold)) if n_features_per_fold else 0.0,
        "skipped_folds": int(skipped_folds),
        "accuracy_mean": float(np.mean(accuracy_scores)) if accuracy_scores else np.nan,
        "accuracy_std": float(np.std(accuracy_scores)) if accuracy_scores else np.nan,
        "f1_mean": float(np.mean(f1_scores)) if f1_scores else np.nan,
        "f1_std": float(np.std(f1_scores)) if f1_scores else np.nan,
        "f1_macro_mean": float(np.mean(f1_macro_scores)) if f1_macro_scores else np.nan,
        "f1_macro_std": float(np.std(f1_macro_scores)) if f1_macro_scores else np.nan,
        "balanced_accuracy_mean": float(np.mean(balanced_accuracy_scores)) if balanced_accuracy_scores else np.nan,
        "balanced_accuracy_std": float(np.std(balanced_accuracy_scores)) if balanced_accuracy_scores else np.nan,
        "mcc_mean": float(np.mean(mcc_scores)) if mcc_scores else np.nan,
        "mcc_std": float(np.std(mcc_scores)) if mcc_scores else np.nan,
    }


def evaluate_thresholds_no_leak(
    X,
    y,
    groups,
    device,
    window,
    overlap_mode,
    start=0.0000,
    end=0.0101,
    step=0.0001,
    smote_method=False,
    n_splits=10,
    n_jobs=1,
    verbose=10,
):
    thresholds = np.arange(start, end, step)

    print(
        f"Evaluando {len(thresholds)} thresholds de {start:.4f} a {end - step:.4f} "
        f"| overlap_mode={overlap_mode} | smote={smote_method} | n_splits={n_splits} | n_jobs={n_jobs}"
    )

    threshold_results = Parallel(n_jobs=n_jobs, verbose=verbose)(
        delayed(evaluate_single_threshold)(
            threshold,
            X,
            y,
            groups,
            smote_method=smote_method,
            n_splits=n_splits,
        )
        for threshold in thresholds
    )

    df = pd.DataFrame(threshold_results)
    df = df.sort_values("f1_mean", ascending=False).reset_index(drop=True)

    if not df.empty:
        print(
            f"Mejor threshold provisional: {df.iloc[0]['threshold']:.4f} "
            f"| f1_mean={df.iloc[0]['f1_mean']:.4f} "
            f"| f1_macro={df.iloc[0]['f1_macro_mean']:.4f} "
            f"| mcc={df.iloc[0]['mcc_mean']:.4f} "
            f"| balanced_accuracy={df.iloc[0]['balanced_accuracy_mean']:.4f} "
            f"| n_features_mean={df.iloc[0]['n_features_mean']:.1f}"
        )

    output_dir = os.path.join(
        get_features_root(overlap_mode=overlap_mode),
        f"features_extracted_{device}",
        "features_benchmark",
        str(window),
    )
    os.makedirs(output_dir, exist_ok=True)

    smote_name = str(smote_method)
    results_path = os.path.join(
        output_dir,
        f"gini_threshold_results_no_leak_{smote_name}.csv",
    )
    df.to_csv(results_path, index=False)

    return df, results_path


In [ ]:
df_results, results_path = evaluate_thresholds_no_leak(
    X,
    y,
    groups,
    device=device,
    window=window,
    overlap_mode=overlap_mode,
    start=threshold_start,
    end=threshold_end,
    step=threshold_step,
    smote_method=smote_method,
    n_splits=n_splits,
    n_jobs=threshold_n_jobs,
)

print("Resultados guardados en:", results_path)
print("\n===== MEJORES RESULTADOS =====")
print(df_results.head())


In [ ]:
smote_name = str(smote_method)
output_dir = os.path.join(
    get_features_root(overlap_mode=overlap_mode),
    f"features_extracted_{device}",
    "features_benchmark",
    str(window),
)
results_path = os.path.join(
    output_dir,
    f"gini_threshold_results_no_leak_{smote_name}.csv",
)

df_results = pd.read_csv(results_path)
best_row = df_results.iloc[0]
best_threshold = float(best_row["threshold"])
best_f1_mean = float(best_row["f1_mean"])

plot_df = df_results.sort_values("threshold").reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(
    plot_df["threshold"],
    plot_df["f1_mean"],
    marker="o",
    markersize=4,
    linewidth=1.6,
    color="tab:blue",
    label="F1 weighted mean",
)
ax1.axvline(
    best_threshold,
    color="tab:blue",
    linestyle="--",
    linewidth=1.4,
    alpha=0.9,
)
ax1.scatter(
    [best_threshold],
    [best_f1_mean],
    color="tab:red",
    s=45,
    zorder=4,
    label=f"Best threshold = {best_threshold:.4f}",
)
ax1.annotate(
    f"{best_f1_mean:.4f}",
    (best_threshold, best_f1_mean),
    textcoords="offset points",
    xytext=(8, 8),
    fontsize=9,
)
ax1.set_xlabel("Threshold")
ax1.set_ylabel("F1 weighted mean", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(True, linestyle="--", alpha=0.6)

ax2 = ax1.twinx()
ax2.plot(
    plot_df["threshold"],
    plot_df["n_features_mean"],
    marker="s",
    markersize=3.5,
    color="tab:orange",
    linewidth=1.3,
    alpha=0.9,
    label="Nº medio de features",
)
ax2.set_ylabel("Nº medio de features", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="best", frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
smote_name = str(smote_method)
output_dir = os.path.join(
    get_features_root(overlap_mode=overlap_mode),
    f"features_extracted_{device}",
    "features_benchmark",
    str(window),
)
results_path = os.path.join(
    output_dir,
    f"gini_threshold_results_no_leak_{smote_name}.csv",
)

df_results = pd.read_csv(results_path)
best_row = df_results.iloc[0]

print("Best row:\n", best_row)
print("\nBest threshold:", float(best_row["threshold"]))
print("Best f1_mean:", float(best_row["f1_mean"]))
print("Best f1_macro_mean:", float(best_row["f1_macro_mean"]))


In [ ]:
smote_name = str(smote_method)
output_dir = os.path.join(
    get_features_root(overlap_mode=overlap_mode),
    f"features_extracted_{device}",
    "features_benchmark",
    str(window),
)
results_path = os.path.join(
    output_dir,
    f"gini_threshold_results_no_leak_{smote_name}.csv",
)

df_results = pd.read_csv(results_path)
best_row = df_results.iloc[0]
best_threshold = float(best_row["threshold"])

X_cv, y_cv, groups_cv = load_data(device=device, window=window, overlap_mode=overlap_mode)
cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
labels = sorted(y_cv.astype(str).unique())

for fold, (train_idx, test_idx) in enumerate(cv.split(X_cv, y_cv, groups_cv), start=1):
    X_train, X_test = X_cv.iloc[train_idx], X_cv.iloc[test_idx]
    y_train, y_test = y_cv.iloc[train_idx], y_cv.iloc[test_idx]
    groups_train = groups_cv.iloc[train_idx]
    groups_test = groups_cv.iloc[test_idx]

    if smote_method:
        X_train_sel, y_train_sel = apply_smote_per_group(
            X_train,
            y_train,
            groups_train,
            method=smote_method,
        )
    else:
        X_train_sel, y_train_sel = X_train, y_train

    selector = RandomForestClassifier(
        n_estimators=500,
        n_jobs=1,
        random_state=42,
    )
    selector.fit(X_train_sel, y_train_sel)

    fold_importance = pd.Series(
        selector.feature_importances_,
        index=X_train_sel.columns,
    )
    feature_list = fold_importance[fold_importance > best_threshold].index.tolist()

    if len(feature_list) == 0:
        print(f"Split {fold}: sin features seleccionadas con threshold {best_threshold:.4f}")
        continue

    model = RandomForestClassifier(
        n_estimators=500,
        n_jobs=1,
        random_state=42,
    )
    model.fit(X_train_sel[feature_list], y_train_sel)
    y_pred = model.predict(X_test[feature_list])

    cm = confusion_matrix(
        y_test.astype(str),
        pd.Series(y_pred).astype(str),
        labels=labels,
    )

    test_users = sorted(pd.Series(groups_test).astype(str).unique())
    print(
        f"\nSplit {fold} | Usuario(s) test: {test_users} | "
        f"f1_mean={f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}"
    )
    print(pd.DataFrame(cm, index=labels, columns=labels))

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_title(f"Split {fold} | User(s): {', '.join(test_users)}")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


In [ ]:
# Final feature selection on the full dataset is only used to train the final artifact
# after choosing the threshold with leakage-free CV.

smote_name = str(smote_method)
output_dir = os.path.join(
    get_features_root(overlap_mode=overlap_mode),
    f"features_extracted_{device}",
    "features_benchmark",
    str(window),
)
os.makedirs(output_dir, exist_ok=True)

results_path = os.path.join(
    output_dir,
    f"gini_threshold_results_no_leak_{smote_name}.csv",
)
df_results = pd.read_csv(results_path)
best_row = df_results.iloc[0]
best_threshold = float(best_row["threshold"])

X_final, y_final, groups_final = load_data(device=device, window=window, overlap_mode=overlap_mode)

if smote_method:
    X_selector, y_selector = apply_smote_per_group(
        X_final,
        y_final,
        groups_final,
        method=smote_method,
    )
else:
    X_selector, y_selector = X_final, y_final

selector = RandomForestClassifier(
    n_estimators=500,
    n_jobs=4,
    random_state=42,
)
selector.fit(X_selector, y_selector)

gini_importance = pd.Series(
    selector.feature_importances_,
    index=X_selector.columns,
    name="gini_importance",
).sort_values(ascending=False)

best_features = gini_importance[gini_importance > best_threshold].index.tolist()

selected_metrics = {
    "accuracy_mean": float(best_row["accuracy_mean"]),
    "accuracy_std": float(best_row["accuracy_std"]),
    "f1_mean": float(best_row["f1_mean"]),
    "f1_std": float(best_row["f1_std"]),
    "f1_macro_mean": float(best_row["f1_macro_mean"]),
    "f1_macro_std": float(best_row["f1_macro_std"]),
    "n_features_mean": float(best_row["n_features_mean"]),
    "n_features_std": float(best_row["n_features_std"]),
    "skipped_folds": int(best_row["skipped_folds"]),
}

for metric_name in ["balanced_accuracy", "mcc"]:
    for statistic in ["mean", "std"]:
        key = f"{metric_name}_{statistic}"
        if key in best_row:
            selected_metrics[key] = float(best_row[key])

gini_path = os.path.join(
    output_dir,
    f"gini_importance_no_leak_{smote_name}.csv",
)
gini_importance.to_csv(gini_path, header=True)

features_payload = {
    "best_threshold": best_threshold,
    "best_features": best_features,
    "selection_metrics": selected_metrics,
}

features_path = os.path.join(
    output_dir,
    f"gini_best_features_no_leak_{smote_name}.json",
)
with open(features_path, "w", encoding="utf-8") as f:
    json.dump(features_payload, f, indent=4)

print("Numero de features seleccionadas:", len(best_features))
print(best_features[:20])
print("\nGini guardado en:", gini_path)
print("Features guardadas en:", features_path)


In [ ]:
smote_name = str(smote_method)
output_dir = os.path.join(
    get_features_root(overlap_mode=overlap_mode),
    f"features_extracted_{device}",
    "features_benchmark",
    str(window),
)

gini_path = os.path.join(
    output_dir,
    f"gini_importance_no_leak_{smote_name}.csv",
)
features_path = os.path.join(
    output_dir,
    f"gini_best_features_no_leak_{smote_name}.json",
)

gini_importance = pd.read_csv(gini_path, index_col=0).iloc[:, 0]
gini_importance.name = "gini_importance"

with open(features_path, "r", encoding="utf-8") as f:
    features_payload = json.load(f)

threshold = float(features_payload["best_threshold"])

# ======================
# Plot superpuesto por sensor
# ======================
gsr = gini_importance[gini_importance.index.str.startswith("Gsr__")].sort_values(ascending=False)
hr = gini_importance[gini_importance.index.str.startswith("Hr__")].sort_values(ascending=False)

n_gsr_selected = int((gsr >= threshold).sum())
n_hr_selected = int((hr >= threshold).sum())

fig, ax = plt.subplots(figsize=(8, 4))

x_gsr = np.arange(len(gsr))
x_hr = np.arange(len(hr))

ax.bar(
    x_gsr,
    gsr.values,
    width=1.0,
    align="edge",
    color="tab:blue",
    alpha=0.62,
    edgecolor="none",
    linewidth=0,
    label="Gsr"
)

ax.bar(
    x_hr,
    hr.values,
    width=1.0,
    align="edge",
    color="tab:orange",
    alpha=0.62,
    edgecolor="none",
    linewidth=0,
    label="Hr"
)

ax.axhline(
    threshold,
    color="black",
    linestyle="--",
    linewidth=1.4,
    alpha=0.9
)

if n_gsr_selected > 0:
    ax.axvline(
        n_gsr_selected,
        color="tab:blue",
        linestyle="-",
        linewidth=2.2,
        alpha=0.95
    )

if n_hr_selected > 0:
    ax.axvline(
        n_hr_selected,
        color="tab:orange",
        linestyle="-",
        linewidth=2.2,
        alpha=0.95
    )

if n_gsr_selected > 0:
    ax.axvspan(
        0,
        n_gsr_selected,
        color="tab:blue",
        alpha=0.08
    )

if n_hr_selected > 0:
    ax.axvspan(
        0,
        n_hr_selected,
        color="tab:orange",
        alpha=0.08
    )

ymax = max(gsr.max() if len(gsr) else 0, hr.max() if len(hr) else 0)
ax.set_ylim(0, ymax * 1.18)

ax.text(
    0.99,
    threshold,
    f"{threshold:.4f}",
    transform=ax.get_yaxis_transform(),
    ha="right",
    va="bottom",
    fontsize=9,
    color="black"
)

if n_gsr_selected > 0:
    ax.text(
        n_gsr_selected,
        ymax * 1.10,
        f"{n_gsr_selected} Gsr",
        ha="left",
        va="bottom",
        fontsize=9,
        color="tab:blue",
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="tab:blue",
            alpha=0.9
        )
    )

if n_hr_selected > 0:
    ax.text(
        n_hr_selected,
        ymax * 1.02,
        f"{n_hr_selected} Hr",
        ha="left",
        va="bottom",
        fontsize=9,
        color="tab:orange",
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="tab:orange",
            alpha=0.9
        )
    )

max_len = max(len(gsr), len(hr))
ax.set_xlim(0, max_len)
ax.margins(x=0)

ax.set_xticks([])
ax.set_ylabel("Importancia (Gini)")
ax.set_xlabel("Característica")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_elements = [
    Patch(facecolor="tab:blue", alpha=0.62, label="Gsr"),
    Patch(facecolor="tab:orange", alpha=0.62, label="Hr")
]

ax.legend(
    handles=legend_elements,
    loc="upper right",
    frameon=True
)

plt.tight_layout()
pdf_path = os.path.join(
    output_dir,
    f"gini_features_{device}_window_{window}_no_leak_{smote_name}_{overlap_mode}.pdf",
)
plt.savefig(
    pdf_path,
    format="pdf",
    bbox_inches="tight"
)
print("PDF guardado en:", pdf_path)
plt.show()


In [ ]:
smote_name = str(smote_method)
output_dir = os.path.join(
    get_features_root(overlap_mode=overlap_mode),
    f"features_extracted_{device}",
    "features_benchmark",
    str(window),
)
features_path = os.path.join(
    output_dir,
    f"gini_best_features_no_leak_{smote_name}.json",
)

with open(features_path, "r", encoding="utf-8") as f:
    features_payload = json.load(f)

print("Features cargadas desde:", features_path)
print("Best threshold:", features_payload["best_threshold"])
print("Numero de features:", len(features_payload["best_features"]))
print(features_payload["best_features"][:20])
print("\nMetricas guardadas:")
for key, value in features_payload["selection_metrics"].items():
    print(f"  {key}: {value}")
